# Earnings Edge Validator — Test EVERYTHING

**Goal**: Test every possible hypothesis on earnings events to find ANY repeatable pattern, no matter what form it takes.

**No Assumptions — Test:**
- **Time periods**: Intraday (open→close same day), overnight, T+1, T+2, T+3, T+5, T+10, T+20
- **Directions**: Long, Short, Long-Short pairs, Market-neutral
- **Entry/Exit**: Open entry, Close entry, Next-day entry, Stop-loss exits, Time exits
- **Signals**: EPS surprise (pos/neg), Gap size, Surprise magnitude, Session timing, Sector, Market cap
- **Filters**: Volume spike, EMA regime (bull/bear), Volatility, Previous momentum
- **Combos**: Surprise + Gap, Surprise + Regime, Gap + Volume, Triple combos, etc.

**Method**: Systematic sweep → measure every combination → rank by Sharpe/hit-rate/consistency → find what works.

**Outcome**: Build a companion that shows you high-probability setups based on what the data actually proves, not assumptions.

---

## 1. Environment + Load Latest Earnings Artifacts

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path('/workspaces/quantum-ai-trader_v1.1')
sys.path.insert(0, str(repo_root))

pd.options.display.max_columns = 50
pd.options.display.width = 120
sns.set_style('whitegrid')

print("✅ Environment ready")
print(f"Repo: {repo_root}")

✅ Environment ready
Repo: /workspaces/quantum-ai-trader_v1.1


In [11]:
# Load latest timing-aware earnings event study run
runs_dir = repo_root / 'data' / 'earnings_event_study_runs'

def get_latest_run_with_file(base_dir, filename):
    """Find most recent run folder that contains the target file."""
    subdirs = sorted([d for d in base_dir.iterdir() if d.is_dir()], reverse=True)
    for d in subdirs:
        if (d / filename).exists():
            return d
    return None

latest_run = get_latest_run_with_file(runs_dir, 'events.csv')
if not latest_run:
    raise FileNotFoundError("No earnings event study runs found with events.csv")

events_path = latest_run / 'events.csv'
summary_path = latest_run / 'summary_by_eps_surprise.csv'

print(f"📂 Loading artifacts from: {latest_run.name}")
events = pd.read_csv(events_path)
summary = pd.read_csv(summary_path) if summary_path.exists() else None

# Count unique tickers to see universe size
unique_tickers = events['ticker'].nunique() if 'ticker' in events.columns else 0

print(f"Events loaded: {len(events)} rows from {unique_tickers} unique tickers")
print(f"Columns: {list(events.columns[:15])}...")
print(f"\nFirst 3 rows:")
events.head(3)

📂 Loading artifacts from: 20251218_003239
Events loaded: 5646 rows from 268 unique tickers
Columns: ['ticker', 'benchmark', 'earnings_datetime', 'earnings_date', 'earnings_session', 'prev_trading_day', 'event_trading_day', 'event_plus_5d', 'prev_close', 'event_open', 'event_close', 'event_plus_5d_close', 'gap_prevclose_to_open', 'ret_prevclose_to_close_1d', 'ret_prevclose_to_close_5d']...

First 3 rows:


,ticker,benchmark,earnings_datetime,earnings_date,earnings_session,prev_trading_day,event_trading_day,event_plus_5d,prev_close,event_open,event_close,event_plus_5d_close,gap_prevclose_to_open,ret_prevclose_to_close_1d,ret_prevclose_to_close_5d,ret_open_to_close_1d,ret_open_to_close_5d,bm_gap_prevclose_to_open,bm_ret_prevclose_to_close_1d,bm_ret_prevclose_to_close_5d,ab_gap_prevclose_to_open,ab_ret_prevclose_to_close_1d,ab_ret_prevclose_to_close_5d,eps_estimate,eps_reported,eps_surprise,positive_eps_surprise
0,AAPL,SPY,2025-10-30 16:00:00-04:00,2025-10-30,after_close,2025-10-29,2025-10-31,2025-11-07,269.700012,276.989990,270.369995,268.470001,0.027030,0.002484,-0.004561,-0.023900,-0.030759,-0.003419,-0.007754,-0.023888,0.030449,0.010238,0.019327,1.77,1.85,0.08,True
1,AAPL,SPY,2025-07-31 16:00:00-04:00,2025-07-31,after_close,2025-07-30,2025-08-01,2025-08-08,209.050003,210.869995,202.380005,229.350006,0.008706,-0.031906,0.097106,-0.040262,0.087637,-0.012861,-0.020080,0.004287,0.021567,-0.011826,0.092819,1.43,1.57,0.14,True
2,AAPL,SPY,2025-05-01 16:00:00-04:00,2025-05-01,after_close,2025-04-30,2025-05-02,2025-05-09,212.500000,206.089996,205.350006,198.529999,-0.030165,-0.033647,-0.065741,-0.003591,-0.036683,0.018376,0.022036,0.017672,-0.048540,-0.055683,-0.083414,1.62,1.65,0.03,True


## 2. Comprehensive Hypothesis Sweep — All Time Periods × All Directions

Test matrix:
- **Horizons**: Same-day (open→close), T+1, T+2, T+3, T+5, T+10, T+20
- **Signals**: Positive surprise, Negative surprise, Big gap (>2%), Big surprise (top 25%)
- **Directions**: LONG, SHORT
- **Measure**: Net mean return, Hit rate, Sharpe (trade-level), Sample size

No assumptions — just run every combination and see what survives costs.

In [12]:
# Prepare dataset: focus on after-close earnings (656 events from timing-aware run)
df = events[events['positive_eps_surprise'].notna()].copy()

# Parse all return columns
ret_cols = [c for c in df.columns if 'ret_' in c or 'gap_' in c]
for col in ret_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Compute derived signals
df['gap_abs'] = df['gap_prevclose_to_open'].abs() if 'gap_prevclose_to_open' in df.columns else 0
df['eps_surprise'] = pd.to_numeric(df.get('eps_surprise', 0), errors='coerce')
df['surprise_abs'] = df['eps_surprise'].abs()

# Define thresholds
gap_thresh = 0.02  # 2%
surprise_pct_cutoff = 75  # top 25%
surprise_cutoff = df['surprise_abs'].quantile(surprise_pct_cutoff / 100) if df['surprise_abs'].notna().any() else 0

print(f"Total events with EPS coverage: {len(df)}")
print(f"  Positive surprise: {(df['positive_eps_surprise']==True).sum()}")
print(f"  Negative surprise: {(df['positive_eps_surprise']==False).sum()}")
print(f"  Gap threshold: {gap_thresh*100:.1f}%")
print(f"  Surprise cutoff (75th pct): {surprise_cutoff:.4f}")

Total events with EPS coverage: 5542
  Positive surprise: 3736
  Negative surprise: 1806
  Gap threshold: 2.0%
  Surprise cutoff (75th pct): 0.2700


In [4]:
# Build comprehensive test matrix
# Test every signal × every horizon × long/short

cost_bps = 5.0
roundtrip_cost = 2 * cost_bps / 10000

def test_strategy(df_input, signal_filter, ret_col, direction='LONG', label=''):
    """
    Test a trading strategy.
    
    Args:
        df_input: DataFrame with events
        signal_filter: Boolean mask for entry signal
        ret_col: Column name for returns
        direction: 'LONG' or 'SHORT'
        label: Strategy description
    
    Returns: Dict with performance metrics
    """
    if ret_col not in df_input.columns:
        return None
    
    signals = df_input[signal_filter].copy()
    if len(signals) == 0:
        return None
    
    rets = pd.to_numeric(signals[ret_col], errors='coerce').dropna()
    if len(rets) == 0:
        return None
    
    # Apply direction
    if direction == 'SHORT':
        rets = -rets
    
    net = rets - roundtrip_cost
    
    # Sharpe-like metric (trade-level, not time-series)
    sharpe_trades = (net.mean() / net.std()) if net.std() > 0 else 0
    
    return {
        'strategy': label,
        'signal': signal_filter.sum(),
        'n': len(net),
        'direction': direction,
        'gross_mean': float(rets.mean()),
        'net_mean': float(net.mean()),
        'net_median': float(net.median()),
        'hit_rate': float((net > 0).mean()),
        'sharpe_trades': float(sharpe_trades),
        'best': float(net.max()),
        'worst': float(net.min()),
        'std': float(net.std()),
    }

print("✅ Strategy tester ready")

✅ Strategy tester ready


In [13]:
# SWEEP 1: All time horizons × Positive Surprise (LONG and SHORT)
horizons = [
    ('ret_open_to_close_1d', 'Same-day'),
    ('ret_open_to_close_5d', 'T+5'),
    ('ret_prevclose_to_close_1d', 'Overnight+T0'),
    ('ret_prevclose_to_close_5d', 'Overnight+T5'),
]

results_matrix = []

# Test POS surprise
pos_mask = df['positive_eps_surprise'] == True

for ret_col, horizon_label in horizons:
    # Long
    r = test_strategy(df, pos_mask, ret_col, 'LONG', f'POS → LONG {horizon_label}')
    if r: results_matrix.append(r)
    
    # Short
    r = test_strategy(df, pos_mask, ret_col, 'SHORT', f'POS → SHORT {horizon_label}')
    if r: results_matrix.append(r)

# Test NEG surprise
neg_mask = df['positive_eps_surprise'] == False

for ret_col, horizon_label in horizons:
    # Long (contrarian bounce)
    r = test_strategy(df, neg_mask, ret_col, 'LONG', f'NEG → LONG {horizon_label}')
    if r: results_matrix.append(r)
    
    # Short (momentum)
    r = test_strategy(df, neg_mask, ret_col, 'SHORT', f'NEG → SHORT {horizon_label}')
    if r: results_matrix.append(r)

sweep1 = pd.DataFrame(results_matrix)
print("\n=== SWEEP 1: All Horizons × Positive/Negative Surprise × Long/Short ===")
print(sweep1[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].sort_values('net_mean', ascending=False).to_string(index=False))
print("\nTop 5 by net_mean:")


=== SWEEP 1: All Horizons × Positive/Negative Surprise × Long/Short ===
                strategy    n  net_mean  hit_rate  sharpe_trades
NEG → SHORT Overnight+T5 1806  0.022072  0.594131       0.125425
NEG → SHORT Overnight+T0 1806  0.021512  0.601883       0.147456
 POS → LONG Overnight+T5 3736  0.019449  0.509636       0.107506
 POS → LONG Overnight+T0 3736  0.015641  0.531585       0.124598
          NEG → LONG T+5 1806  0.003264  0.473422       0.021385
     NEG → LONG Same-day 1806  0.003161  0.488372       0.036249
          POS → LONG T+5 3736  0.002292  0.465471       0.016099
    POS → SHORT Same-day 3736 -0.000997  0.521146      -0.014099
     POS → LONG Same-day 3736 -0.001003  0.464668      -0.014189
         POS → SHORT T+5 3736 -0.004292  0.524893      -0.030150
    NEG → SHORT Same-day 1806 -0.005161  0.496124      -0.059184
         NEG → SHORT T+5 1806 -0.005264  0.515504      -0.034488
POS → SHORT Overnight+T0 3736 -0.017641  0.459315      -0.140531
POS → SHORT Overn

In [14]:
# SWEEP 2: Add filters — Big Gap, Big Surprise, Session Timing
results_filtered = []

# Filter: Big gap (>2%)
big_gap_mask = df['gap_abs'] > gap_thresh

for ret_col, horizon_label in [('ret_open_to_close_5d', 'T+5')]:
    # POS + big gap → LONG
    r = test_strategy(df, pos_mask & big_gap_mask, ret_col, 'LONG', f'POS + BigGap → LONG {horizon_label}')
    if r: results_filtered.append(r)
    
    # NEG + big gap → LONG (contrarian)
    r = test_strategy(df, neg_mask & big_gap_mask, ret_col, 'LONG', f'NEG + BigGap → LONG {horizon_label}')
    if r: results_filtered.append(r)
    
    # NEG + big gap → SHORT (momentum)
    r = test_strategy(df, neg_mask & big_gap_mask, ret_col, 'SHORT', f'NEG + BigGap → SHORT {horizon_label}')
    if r: results_filtered.append(r)

# Filter: Big surprise magnitude
if surprise_cutoff > 0:
    big_surprise_mask = df['surprise_abs'] > surprise_cutoff
    
    for ret_col, horizon_label in [('ret_open_to_close_5d', 'T+5')]:
        r = test_strategy(df, pos_mask & big_surprise_mask, ret_col, 'LONG', f'POS + BigSurprise → LONG {horizon_label}')
        if r: results_filtered.append(r)
        
        r = test_strategy(df, neg_mask & big_surprise_mask, ret_col, 'LONG', f'NEG + BigSurprise → LONG {horizon_label}')
        if r: results_filtered.append(r)

# Filter: Session timing (after_close vs before_open)
if 'earnings_session' in df.columns:
    after_close_mask = df['earnings_session'] == 'after_close'
    before_open_mask = df['earnings_session'] == 'before_open'
    
    for ret_col, horizon_label in [('ret_open_to_close_5d', 'T+5')]:
        r = test_strategy(df, pos_mask & after_close_mask, ret_col, 'LONG', f'POS + AfterClose → LONG {horizon_label}')
        if r: results_filtered.append(r)
        
        r = test_strategy(df, pos_mask & before_open_mask, ret_col, 'LONG', f'POS + BeforeOpen → LONG {horizon_label}')
        if r: results_filtered.append(r)

# Combo: POS + big gap + big surprise
if surprise_cutoff > 0:
    triple_mask = pos_mask & big_gap_mask & big_surprise_mask
    r = test_strategy(df, triple_mask, 'ret_open_to_close_5d', 'LONG', 'POS + BigGap + BigSurprise → LONG T+5')
    if r: results_filtered.append(r)

sweep2 = pd.DataFrame(results_filtered)
print("\n=== SWEEP 2: Filtered Strategies (Gap, Surprise Magnitude, Session) ===")
print(sweep2[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].sort_values('net_mean', ascending=False).to_string(index=False))


=== SWEEP 2: Filtered Strategies (Gap, Surprise Magnitude, Session) ===
                             strategy    n  net_mean  hit_rate  sharpe_trades
              NEG + BigGap → LONG T+5 1337  0.006785  0.481675       0.042432
              POS + BigGap → LONG T+5 2858  0.005299  0.475157       0.035477
          POS + BeforeOpen → LONG T+5 1187  0.003764  0.466723       0.023447
         NEG + BigSurprise → LONG T+5  523  0.002575  0.487572       0.017969
          POS + AfterClose → LONG T+5 2530  0.001680  0.465613       0.012623
         POS + BigSurprise → LONG T+5  846 -0.000619  0.442080      -0.003798
POS + BigGap + BigSurprise → LONG T+5  635 -0.003150  0.437795      -0.018195
             NEG + BigGap → SHORT T+5 1337 -0.008785  0.509349      -0.054939


In [15]:
# SWEEP 3: Mean reversion vs Momentum — test opposite bets
# Hypothesis: Maybe the gap REVERSES (fade), or maybe it CONTINUES (momentum)
results_reversion = []

# Gap reversal: big gap → bet against it
for ret_col, horizon_label in [('ret_open_to_close_1d', 'Same-day'), ('ret_open_to_close_5d', 'T+5')]:
    # Big gap UP → SHORT (fade)
    big_gap_up = (df['gap_prevclose_to_open'] > gap_thresh) if 'gap_prevclose_to_open' in df.columns else pd.Series(False, index=df.index)
    r = test_strategy(df, big_gap_up, ret_col, 'SHORT', f'BigGapUP → SHORT (fade) {horizon_label}')
    if r: results_reversion.append(r)
    
    # Big gap DOWN → LONG (bounce)
    big_gap_down = (df['gap_prevclose_to_open'] < -gap_thresh) if 'gap_prevclose_to_open' in df.columns else pd.Series(False, index=df.index)
    r = test_strategy(df, big_gap_down, ret_col, 'LONG', f'BigGapDOWN → LONG (bounce) {horizon_label}')
    if r: results_reversion.append(r)

# Gap momentum: big gap → bet with it
for ret_col, horizon_label in [('ret_open_to_close_5d', 'T+5')]:
    # Big gap UP → LONG (momentum)
    r = test_strategy(df, big_gap_up, ret_col, 'LONG', f'BigGapUP → LONG (momentum) {horizon_label}')
    if r: results_reversion.append(r)
    
    # Big gap DOWN → SHORT (momentum)
    r = test_strategy(df, big_gap_down, ret_col, 'SHORT', f'BigGapDOWN → SHORT (momentum) {horizon_label}')
    if r: results_reversion.append(r)

sweep3 = pd.DataFrame(results_reversion)
print("\n=== SWEEP 3: Mean Reversion vs Momentum (Gap Fade/Continue) ===")
print(sweep3[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].sort_values('net_mean', ascending=False).to_string(index=False))


=== SWEEP 3: Mean Reversion vs Momentum (Gap Fade/Continue) ===
                           strategy    n  net_mean  hit_rate  sharpe_trades
     BigGapUP → LONG (momentum) T+5 2104  0.006348  0.490494       0.047753
     BigGapDOWN → LONG (bounce) T+5 2091  0.005193  0.463893       0.030466
BigGapDOWN → LONG (bounce) Same-day 2091  0.001292  0.476327       0.016424
   BigGapUP → SHORT (fade) Same-day 2104 -0.003107  0.516635      -0.037630
  BigGapDOWN → SHORT (momentum) T+5 2091 -0.007193  0.526542      -0.042198
        BigGapUP → SHORT (fade) T+5 2104 -0.008348  0.500951      -0.062798


## 3. Aggregate & Rank — What Actually Works?

Combine all test results and rank by:
1. **Net mean return** (profit after costs)
2. **Hit rate** (consistency)
3. **Sharpe (trade-level)** (risk-adjusted)
4. **Sample size** (robustness)

Filter: Keep only strategies with n ≥ 50 and net_mean > 0

In [16]:
# Combine all sweeps
all_results = pd.concat([sweep1, sweep2, sweep3], ignore_index=True)

# Filter: reasonable sample size + positive net return
viable = all_results[(all_results['n'] >= 50) & (all_results['net_mean'] > 0)].copy()

# Rank by net_mean
viable = viable.sort_values(['net_mean', 'hit_rate'], ascending=False)

print("="*100)
print("TOP VIABLE STRATEGIES (n≥50, net_mean>0, sorted by net_mean)")
print("="*100)
print(viable[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades', 'net_median']].head(20).to_string(index=False))

# Also show by Sharpe
viable_sharpe = viable.sort_values('sharpe_trades', ascending=False)
print("\\n" + "="*100)
print("TOP BY SHARPE (trade-level)")
print("="*100)
print(viable_sharpe[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].head(10).to_string(index=False))

TOP VIABLE STRATEGIES (n≥50, net_mean>0, sorted by net_mean)
                           strategy    n  net_mean  hit_rate  sharpe_trades  net_median
           NEG → SHORT Overnight+T5 1806  0.022072  0.594131       0.125425    0.032272
           NEG → SHORT Overnight+T0 1806  0.021512  0.601883       0.147456    0.022260
            POS → LONG Overnight+T5 3736  0.019449  0.509636       0.107506    0.003194
            POS → LONG Overnight+T0 3736  0.015641  0.531585       0.124598    0.006645
            NEG + BigGap → LONG T+5 1337  0.006785  0.481675       0.042432   -0.005428
     BigGapUP → LONG (momentum) T+5 2104  0.006348  0.490494       0.047753   -0.002166
            POS + BigGap → LONG T+5 2858  0.005299  0.475157       0.035477   -0.005118
     BigGapDOWN → LONG (bounce) T+5 2091  0.005193  0.463893       0.030466   -0.007936
        POS + BeforeOpen → LONG T+5 1187  0.003764  0.466723       0.023447   -0.007211
                     NEG → LONG T+5 1806  0.003264  0.47342

## 4. Visualize Return Distributions — Winners vs Losers

In [ ]:
# Plot top 4 strategies by net_mean
top_strategies = viable.head(4)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (i, row) in enumerate(top_strategies.iterrows()):
    if idx >= 4:
        break
    
    # Reconstruct the signal for this strategy
    # (This is simplified; in production you'd store the actual trade returns)
    strat_name = row['strategy']
    print(f"Plotting: {strat_name}")
    
    # For visualization: just show POS vs NEG T+5 returns as proxy
    if 'POS' in strat_name:
        sample = df[df['positive_eps_surprise'] == True]['ret_open_to_close_5d'].dropna()
    else:
        sample = df[df['positive_eps_surprise'] == False]['ret_open_to_close_5d'].dropna()
    
    if 'SHORT' in strat_name:
        sample = -sample
    
    sample_net = sample - roundtrip_cost
    
    axes[idx].hist(sample_net * 100, bins=40, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].axvline(sample_net.mean() * 100, color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: {sample_net.mean()*100:.2f}%')
    axes[idx].axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
    axes[idx].set_xlabel('Net Return (%)')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f"{strat_name}\\nn={row['n']}, hit={row['hit_rate']*100:.1f}%")
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary & Next Steps

**What worked? What failed?**

Review the top-ranked strategies and determine:
1. Are there repeatable patterns with net_mean > 1% and hit_rate > 52%?
2. Which time horizons work best?
3. Does filtering (gap, surprise magnitude, session timing) improve edge?
4. Mean reversion vs momentum — which wins?

**Next actions:**
- If we found edges: scale to 300-ticker universe and validate stability
- Add EMA regime filter (bull/bear trend context)
- Add volume confirmation (spike on earnings day)
- Build real-time companion: "upcoming earnings + filters → show high-prob setups"

In [17]:
# Final summary table
print("="*100)
print("EARNINGS EDGE VALIDATION — COMPREHENSIVE SWEEP RESULTS")
print("="*100)
print(f"\\nTotal strategies tested: {len(all_results)}")
print(f"Viable strategies (n≥50, net_mean>0): {len(viable)}")

if len(viable) > 0:
    best = viable.iloc[0]
    print(f"\\n🏆 BEST STRATEGY:")
    print(f"   {best['strategy']}")
    print(f"   Net Mean: {best['net_mean']*100:.2f}%")
    print(f"   Hit Rate: {best['hit_rate']*100:.1f}%")
    print(f"   Sharpe (trades): {best['sharpe_trades']:.2f}")
    print(f"   Sample Size: {int(best['n'])}")
    
    print(f"\\n📊 Top 3 by net_mean:")
    for i, row in viable.head(3).iterrows():
        print(f"   {i+1}. {row['strategy']}: {row['net_mean']*100:.2f}% (n={int(row['n'])}, hit={row['hit_rate']*100:.1f}%)")
else:
    print("\\n⚠️  No viable strategies found with current filters.")
    print("    Try: relaxing n threshold, testing longer horizons, or adding new signal types.")

print("\\n" + "="*100)
print("FULL RESULTS SAVED IN: all_results, viable DataFrames")
print("="*100)

EARNINGS EDGE VALIDATION — COMPREHENSIVE SWEEP RESULTS
\nTotal strategies tested: 30
Viable strategies (n≥50, net_mean>0): 15
\n🏆 BEST STRATEGY:
   NEG → SHORT Overnight+T5
   Net Mean: 2.21%
   Hit Rate: 59.4%
   Sharpe (trades): 0.13
   Sample Size: 1806
\n📊 Top 3 by net_mean:
   16. NEG → SHORT Overnight+T5: 2.21% (n=1806, hit=59.4%)
   14. NEG → SHORT Overnight+T0: 2.15% (n=1806, hit=60.2%)
   7. POS → LONG Overnight+T5: 1.94% (n=3736, hit=51.0%)
\n====================================================================================================
FULL RESULTS SAVED IN: all_results, viable DataFrames


## 6. Advanced Hypotheses — Quality, Streaks, Sectors, Momentum

Test what we HAVEN'T tested yet:
1. **Earnings STREAKS**: Do stocks with 2+ consecutive positive surprises outperform?
2. **Sector edges**: Does AI/tech/energy react differently to earnings than retail/consumer?
3. **Market cap**: Do small/mid-caps have bigger earnings pops than large-caps?
4. **Surprise MAGNITUDE trends**: Are companies with improving surprise % better bets?
5. **Quality filter**: Consistent beaters (3+ positive out of last 4) vs random beaters
6. **Combinations**: POS + streak + sector + market cap

Find the stocks that are WORTH investing in, not just trading noise.

In [18]:
# Load enriched events if available (has sector/industry from universe metadata)
enriched_path = latest_run / 'events_enriched.csv'
if enriched_path.exists():
    df_enriched = pd.read_csv(enriched_path)
    print(f"✅ Loaded enriched events: {len(df_enriched)} rows")
    print(f"   Extra columns: {[c for c in df_enriched.columns if c not in df.columns]}")
else:
    df_enriched = df.copy()
    print("⚠️  No enriched file; using base events")

# Parse dates for chronological sorting
df_enriched['earnings_date_dt'] = pd.to_datetime(df_enriched['earnings_date'], errors='coerce')
df_enriched = df_enriched.sort_values(['ticker', 'earnings_date_dt'])

print(f"\nTotal events: {len(df_enriched)}")
print(f"Unique tickers: {df_enriched['ticker'].nunique()}")
print(f"Date range: {df_enriched['earnings_date_dt'].min()} to {df_enriched['earnings_date_dt'].max()}")

✅ Loaded enriched events: 5646 rows
   Extra columns: ['sector', 'industry', 'market_cap_category']

Total events: 5646
Unique tickers: 268
Date range: 2012-07-31 00:00:00 to 2025-12-10 00:00:00


In [19]:
# HYPOTHESIS 1: Earnings STREAKS — Do consecutive positive surprises predict bigger moves?

# For each ticker, tag each event with how many consecutive positive surprises it has had
def compute_streaks(group):
    """Compute consecutive positive surprise streak for each event."""
    group = group.sort_values('earnings_date_dt')
    group['prev_pos_streak'] = 0
    
    streak = 0
    for idx in group.index:
        group.loc[idx, 'prev_pos_streak'] = streak
        if group.loc[idx, 'positive_eps_surprise'] == True:
            streak += 1
        else:
            streak = 0
    return group

df_with_streaks = df_enriched.groupby('ticker', group_keys=False).apply(compute_streaks).reset_index(drop=True)

# Test: Does a 2+ streak outperform?
streak_results = []

for min_streak in [0, 1, 2, 3]:
    mask = (df_with_streaks['positive_eps_surprise'] == True) & (df_with_streaks['prev_pos_streak'] >= min_streak)
    r = test_strategy(df_with_streaks, mask, 'ret_open_to_close_5d', 'LONG', f'POS + Streak≥{min_streak} → LONG T+5')
    if r: streak_results.append(r)

streak_df = pd.DataFrame(streak_results)
print("="*80)
print("HYPOTHESIS 1: Earnings Streaks (consecutive positive surprises)")
print("="*80)
print(streak_df[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].to_string(index=False))
print("\n📊 Interpretation: Does edge improve with longer streaks?")

HYPOTHESIS 1: Earnings Streaks (consecutive positive surprises)
                 strategy    n  net_mean  hit_rate  sharpe_trades
POS + Streak≥0 → LONG T+5 3736  0.002292  0.465471       0.016099
POS + Streak≥1 → LONG T+5 2683  0.005379  0.475587       0.037394
POS + Streak≥2 → LONG T+5 2058  0.005622  0.478134       0.037686
POS + Streak≥3 → LONG T+5 1634  0.004232  0.477356       0.031008

📊 Interpretation: Does edge improve with longer streaks?


/tmp/ipykernel_2302/1973019003.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_with_streaks = df_enriched.groupby('ticker', group_keys=False).apply(compute_streaks).reset_index(drop=True)


In [20]:
# HYPOTHESIS 2: SECTOR edges — Do tech/AI/energy stocks react better to earnings?

sector_results = []

if 'sector' in df_enriched.columns:
    # Top sectors by count
    top_sectors = df_enriched['sector'].value_counts().head(10).index.tolist()
    
    for sector in top_sectors:
        if pd.isna(sector):
            continue
        
        sector_mask = (df_enriched['sector'] == sector) & (df_enriched['positive_eps_surprise'] == True)
        r = test_strategy(df_enriched, sector_mask, 'ret_open_to_close_5d', 'LONG', f'{sector[:20]} POS → LONG T+5')
        if r and r['n'] >= 30:  # minimum 30 events
            sector_results.append(r)
    
    sector_df = pd.DataFrame(sector_results).sort_values('net_mean', ascending=False)
    print("="*80)
    print("HYPOTHESIS 2: Sector-Specific Earnings Edges (POS surprise only)")
    print("="*80)
    print(sector_df[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))
    print("\n📊 Which sectors have the strongest earnings reaction?")
else:
    print("⚠️  No sector data available in enriched events")

HYPOTHESIS 2: Sector-Specific Earnings Edges (POS surprise only)
                 strategy    n  net_mean  hit_rate
    Crypto POS → LONG T+5   70  0.038269  0.471429
  Consumer POS → LONG T+5   57  0.011630  0.473684
Technology POS → LONG T+5  965  0.010971  0.481865
    Energy POS → LONG T+5  201  0.004902  0.452736
E-commerce POS → LONG T+5  254  0.003115  0.515748
 Materials POS → LONG T+5  101  0.003074  0.485149
   Fintech POS → LONG T+5  102 -0.001507  0.401961
Healthcare POS → LONG T+5 1287 -0.003728  0.450660
Automotive POS → LONG T+5  108 -0.007277  0.407407
    Retail POS → LONG T+5  237 -0.010743  0.438819

📊 Which sectors have the strongest earnings reaction?


In [21]:
# HYPOTHESIS 3: Market Cap — Do small/mid-caps have bigger earnings pops?

mcap_results = []

if 'market_cap_category' in df_enriched.columns:
    for mcap_cat in df_enriched['market_cap_category'].dropna().unique():
        mcap_mask = (df_enriched['market_cap_category'] == mcap_cat) & (df_enriched['positive_eps_surprise'] == True)
        r = test_strategy(df_enriched, mcap_mask, 'ret_open_to_close_5d', 'LONG', f'{mcap_cat} POS → LONG T+5')
        if r and r['n'] >= 30:
            mcap_results.append(r)
    
    mcap_df = pd.DataFrame(mcap_results).sort_values('net_mean', ascending=False)
    print("="*80)
    print("HYPOTHESIS 3: Market Cap Size (small vs mid vs large)")
    print("="*80)
    print(mcap_df[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))
    print("\n📊 Do small-caps outperform on positive earnings?")
else:
    print("⚠️  No market cap data available")

HYPOTHESIS 3: Market Cap Size (small vs mid vs large)
                strategy    n  net_mean  hit_rate
Small Cap POS → LONG T+5 1776  0.003938  0.452140
  Mid Cap POS → LONG T+5  946  0.003083  0.491543
Large Cap POS → LONG T+5  738 -0.001041  0.478320
Micro Cap POS → LONG T+5  276 -0.002103  0.427536

📊 Do small-caps outperform on positive earnings?


In [22]:
# HYPOTHESIS 4: SURPRISE MAGNITUDE trend — Are stocks with IMPROVING surprises better?

# For each ticker, compute the difference between current surprise and previous surprise
def compute_surprise_trend(group):
    """Compute surprise magnitude trend."""
    group = group.sort_values('earnings_date_dt')
    group['surprise_mag'] = group['eps_surprise'].abs()
    group['prev_surprise_mag'] = group['surprise_mag'].shift(1)
    group['surprise_improving'] = group['surprise_mag'] > group['prev_surprise_mag']
    return group

df_with_trends = df_with_streaks.groupby('ticker', group_keys=False).apply(compute_surprise_trend).reset_index(drop=True)

trend_results = []

# Test: POS + improving surprise magnitude
improving_mask = (df_with_trends['positive_eps_surprise'] == True) & (df_with_trends['surprise_improving'] == True)
r = test_strategy(df_with_trends, improving_mask, 'ret_open_to_close_5d', 'LONG', 'POS + Improving surprise → LONG T+5')
if r: trend_results.append(r)

# Test: POS + declining surprise magnitude
declining_mask = (df_with_trends['positive_eps_surprise'] == True) & (df_with_trends['surprise_improving'] == False)
r = test_strategy(df_with_trends, declining_mask, 'ret_open_to_close_5d', 'LONG', 'POS + Declining surprise → LONG T+5')
if r: trend_results.append(r)

trend_df = pd.DataFrame(trend_results)
print("="*80)
print("HYPOTHESIS 4: Surprise Magnitude Trend (improving vs declining)")
print("="*80)
print(trend_df[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))
print("\n📊 Do companies with IMPROVING surprise magnitude outperform?")

HYPOTHESIS 4: Surprise Magnitude Trend (improving vs declining)
                           strategy    n  net_mean  hit_rate
POS + Improving surprise → LONG T+5 1660  0.000609  0.467470
POS + Declining surprise → LONG T+5 2076  0.003637  0.463873

📊 Do companies with IMPROVING surprise magnitude outperform?


/tmp/ipykernel_2302/521430177.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_with_trends = df_with_streaks.groupby('ticker', group_keys=False).apply(compute_surprise_trend).reset_index(drop=True)


In [23]:
# HYPOTHESIS 5: QUALITY filter — Consistent beaters (3+ out of last 4) vs one-off beaters

def compute_consistency(group):
    """Compute earnings beat consistency."""
    group = group.sort_values('earnings_date_dt')
    group['beat_count_last4'] = group['positive_eps_surprise'].rolling(window=4, min_periods=1).sum()
    group['is_consistent_beater'] = group['beat_count_last4'] >= 3
    return group

df_with_quality = df_with_trends.groupby('ticker', group_keys=False).apply(compute_consistency).reset_index(drop=True)

quality_results = []

# Test: Consistent beaters (3+ out of last 4)
consistent_mask = (df_with_quality['positive_eps_surprise'] == True) & (df_with_quality['is_consistent_beater'] == True)
r = test_strategy(df_with_quality, consistent_mask, 'ret_open_to_close_5d', 'LONG', 'Consistent Beater (3/4) POS → LONG T+5')
if r: quality_results.append(r)

# Test: Inconsistent beaters
inconsistent_mask = (df_with_quality['positive_eps_surprise'] == True) & (df_with_quality['is_consistent_beater'] == False)
r = test_strategy(df_with_quality, inconsistent_mask, 'ret_open_to_close_5d', 'LONG', 'Inconsistent Beater POS → LONG T+5')
if r: quality_results.append(r)

quality_df = pd.DataFrame(quality_results)
print("="*80)
print("HYPOTHESIS 5: Earnings Quality — Consistent vs One-Off Beaters")
print("="*80)
print(quality_df[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))
print("\n📊 Are CONSISTENT beaters (quality companies) better investments?")

HYPOTHESIS 5: Earnings Quality — Consistent vs One-Off Beaters
                              strategy    n  net_mean  hit_rate
Consistent Beater (3/4) POS → LONG T+5 2629  0.003826  0.475846
    Inconsistent Beater POS → LONG T+5 1107 -0.001353  0.440831

📊 Are CONSISTENT beaters (quality companies) better investments?


/tmp/ipykernel_2302/3965475513.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_with_quality = df_with_trends.groupby('ticker', group_keys=False).apply(compute_consistency).reset_index(drop=True)


In [26]:
# HYPOTHESIS 6: ULTIMATE COMBO — Stack all the best filters

combo_results = []

# Recompute gap_abs if missing
if 'gap_abs' not in df_with_quality.columns and 'gap_prevclose_to_open' in df_with_quality.columns:
    df_with_quality['gap_abs'] = df_with_quality['gap_prevclose_to_open'].abs()

# Combo 1: Consistent beater + streak + big gap
if 'is_consistent_beater' in df_with_quality.columns and 'gap_abs' in df_with_quality.columns:
    ultimate_mask = (
        (df_with_quality['positive_eps_surprise'] == True) &
        (df_with_quality['is_consistent_beater'] == True) &
        (df_with_quality['prev_pos_streak'] >= 1) &
        (df_with_quality['gap_abs'] > 0.02)
    )
    r = test_strategy(df_with_quality, ultimate_mask, 'ret_open_to_close_5d', 'LONG', 'ULTIMATE: Quality + Streak + BigGap → LONG T+5')
    if r: combo_results.append(r)

# Combo 2: Consistent beater + improving surprise
if 'surprise_improving' in df_with_quality.columns:
    combo2_mask = (
        (df_with_quality['positive_eps_surprise'] == True) &
        (df_with_quality['is_consistent_beater'] == True) &
        (df_with_quality['surprise_improving'] == True)
    )
    r = test_strategy(df_with_quality, combo2_mask, 'ret_open_to_close_5d', 'LONG', 'Quality + Improving → LONG T+5')
    if r: combo_results.append(r)

# Combo 3: Add sector if available
if 'sector' in df_with_quality.columns:
    # Find best sector from earlier test
    tech_like = df_with_quality['sector'].str.contains('Tech|Information|Software|Semi|Computer', case=False, na=False)
    combo3_mask = (
        (df_with_quality['positive_eps_surprise'] == True) &
        (df_with_quality['is_consistent_beater'] == True) &
        tech_like
    )
    r = test_strategy(df_with_quality, combo3_mask, 'ret_open_to_close_5d', 'LONG', 'Quality + Tech Sector → LONG T+5')
    if r: combo_results.append(r)

combo_df = pd.DataFrame(combo_results)
print("="*80)
print("HYPOTHESIS 6: ULTIMATE COMBOS (stacking filters)")
print("="*80)
if len(combo_df):
    print(combo_df[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].to_string(index=False))
else:
    print("No combo results (insufficient data or features missing)")
print("\n📊 Can we find a 3%+ edge by combining quality signals?")

HYPOTHESIS 6: ULTIMATE COMBOS (stacking filters)
                                      strategy    n  net_mean  hit_rate  sharpe_trades
ULTIMATE: Quality + Streak + BigGap → LONG T+5 1857  0.009111  0.487884       0.058772
                Quality + Improving → LONG T+5 1233  0.002406  0.478508       0.020566
              Quality + Tech Sector → LONG T+5  867  0.012606  0.486736       0.067581

📊 Can we find a 3%+ edge by combining quality signals?


## 7. Final Verdict — What Actually Works?

Aggregate ALL hypothesis test results and rank the top strategies.

In [27]:
# Combine ALL advanced hypothesis results
all_advanced = []
for df_test in [streak_df, sector_df, mcap_df, trend_df, quality_df, combo_df]:
    if len(df_test):
        all_advanced.append(df_test)

if all_advanced:
    advanced_combined = pd.concat(all_advanced, ignore_index=True)
    advanced_viable = advanced_combined[(advanced_combined['n'] >= 30) & (advanced_combined['net_mean'] > 0)].copy()
    advanced_viable = advanced_viable.sort_values(['net_mean', 'hit_rate'], ascending=False)
    
    print("="*100)
    print("🎯 TOP ADVANCED STRATEGIES (n≥30, net>0, ranked by net_mean)")
    print("="*100)
    print(advanced_viable[['strategy', 'n', 'net_mean', 'hit_rate', 'sharpe_trades']].head(15).to_string(index=False))
    
    # Find the best overall
    if len(advanced_viable) > 0:
        best_advanced = advanced_viable.iloc[0]
        print(f"\n🏆 BEST ADVANCED STRATEGY:")
        print(f"   {best_advanced['strategy']}")
        print(f"   Net Mean: {best_advanced['net_mean']*100:.2f}%")
        print(f"   Hit Rate: {best_advanced['hit_rate']*100:.1f}%")
        print(f"   Sample: {int(best_advanced['n'])}")
        print(f"   Sharpe: {best_advanced['sharpe_trades']:.2f}")
else:
    print("⚠️  No advanced results to combine (check if sector/market cap data is missing)")

🎯 TOP ADVANCED STRATEGIES (n≥30, net>0, ranked by net_mean)
                                      strategy    n  net_mean  hit_rate  sharpe_trades
                         Crypto POS → LONG T+5   70  0.038269  0.471429       0.152242
              Quality + Tech Sector → LONG T+5  867  0.012606  0.486736       0.067581
                       Consumer POS → LONG T+5   57  0.011630  0.473684       0.111868
                     Technology POS → LONG T+5  965  0.010971  0.481865       0.060826
ULTIMATE: Quality + Streak + BigGap → LONG T+5 1857  0.009111  0.487884       0.058772
                     POS + Streak≥2 → LONG T+5 2058  0.005622  0.478134       0.037686
                     POS + Streak≥1 → LONG T+5 2683  0.005379  0.475587       0.037394
                         Energy POS → LONG T+5  201  0.004902  0.452736       0.030315
                     POS + Streak≥3 → LONG T+5 1634  0.004232  0.477356       0.031008
                      Small Cap POS → LONG T+5 1776  0.003938  0.45214

In [ ]:
# Prepare dataset: after-close earnings only (majority of events)
df = events[events['positive_eps_surprise'].notna()].copy()
df = df[df['earnings_session'] == 'after_close'].copy()

# Parse numeric columns
for col in ['ret_open_to_close_1d', 'ret_open_to_close_5d', 'gap_prevclose_to_open']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['ret_open_to_close_1d'])

print(f"After-close earnings with EPS coverage: {len(df)}")
print(f"Positive surprise: {(df['positive_eps_surprise']==True).sum()}")
print(f"Negative/miss: {(df['positive_eps_surprise']==False).sum()}")

In [ ]:
# Baseline strategy: LONG positive surprise, overnight hold
# (T+1: enter next open, exit next close; T+5: enter next open, exit 5 days later)

cost_bps = 5.0  # per side
roundtrip_cost = 2 * cost_bps / 10000

def test_long_strategy(df, signal_col, ret_col, label):
    """Test simple long strategy: buy next open if signal==True, sell later."""
    signals = df[df[signal_col] == True].copy()
    if ret_col not in signals.columns:
        return None
    
    rets = pd.to_numeric(signals[ret_col], errors='coerce').dropna()
    net = rets - roundtrip_cost
    
    return {
        'strategy': label,
        'n': len(net),
        'gross_mean': float(rets.mean()),
        'net_mean': float(net.mean()),
        'hit_rate': float((net > 0).mean()),
        'gross_median': float(rets.median()),
        'net_median': float(net.median()),
        'best': float(net.max()),
        'worst': float(net.min()),
    }

# Test positive surprise → long overnight
results = []
results.append(test_long_strategy(df, 'positive_eps_surprise', 'ret_open_to_close_1d', 'POS → LONG T+1'))
results.append(test_long_strategy(df, 'positive_eps_surprise', 'ret_open_to_close_5d', 'POS → LONG T+5'))

# Test negative surprise → long overnight (contrarian)
df_neg = df.copy()
df_neg['negative_eps_surprise'] = ~df_neg['positive_eps_surprise']
results.append(test_long_strategy(df_neg, 'negative_eps_surprise', 'ret_open_to_close_1d', 'NEG → LONG T+1'))
results.append(test_long_strategy(df_neg, 'negative_eps_surprise', 'ret_open_to_close_5d', 'NEG → LONG T+5'))

baseline = pd.DataFrame([r for r in results if r is not None])
print("\n=== BASELINE: Overnight Holds (costs=5bps/side) ===")
print(baseline[['strategy', 'n', 'net_mean', 'hit_rate', 'net_median']].to_string(index=False))
print("\nNote: PDT-compliant — all holds are overnight minimum.")

## 3. Add Confirmation Filters: Volume + Gap Magnitude

Hypothesis: edge improves if we filter for:
- Large gap (|gap| > threshold)
- High surprise magnitude (|EPS surprise| > threshold)

Test ablation: does filtering improve net mean + hit rate?

In [ ]:
# Compute gap magnitude and surprise magnitude
df['gap_abs'] = df['gap_prevclose_to_open'].abs()
df['eps_surprise'] = pd.to_numeric(df['eps_surprise'], errors='coerce')
df['surprise_abs'] = df['eps_surprise'].abs()

# Filter thresholds
gap_thresh = 0.02  # 2% gap
surprise_thresh_pct = 75  # top 25% by surprise magnitude

surprise_cutoff = df['surprise_abs'].quantile(surprise_thresh_pct / 100) if df['surprise_abs'].notna().any() else None

print(f"Gap threshold: {gap_thresh*100:.1f}%")
print(f"Surprise magnitude cutoff (75th pct): {surprise_cutoff if surprise_cutoff else 'N/A'}")

In [ ]:
# Test: POS surprise + big gap → LONG T+1
def test_filtered_long(df, signal_col, ret_col, filters, label):
    """Test long strategy with additional filters."""
    mask = df[signal_col] == True
    for f in filters:
        mask = mask & f
    
    signals = df[mask].copy()
    if ret_col not in signals.columns or len(signals) == 0:
        return None
    
    rets = pd.to_numeric(signals[ret_col], errors='coerce').dropna()
    net = rets - roundtrip_cost
    
    return {
        'strategy': label,
        'n': len(net),
        'net_mean': float(net.mean()),
        'hit_rate': float((net > 0).mean()),
        'net_median': float(net.median()),
    }

ablation = []

# Baseline (no filter)
ablation.append(test_filtered_long(df, 'positive_eps_surprise', 'ret_open_to_close_5d', [], 'POS → LONG T+5 (no filter)'))

# + Big gap
ablation.append(test_filtered_long(df, 'positive_eps_surprise', 'ret_open_to_close_5d', 
                                   [df['gap_abs'] > gap_thresh], 
                                   'POS + big gap → LONG T+5'))

# + Big surprise
if surprise_cutoff:
    ablation.append(test_filtered_long(df, 'positive_eps_surprise', 'ret_open_to_close_5d', 
                                       [df['surprise_abs'] > surprise_cutoff], 
                                       'POS + big surprise → LONG T+5'))

# + Both
if surprise_cutoff:
    ablation.append(test_filtered_long(df, 'positive_eps_surprise', 'ret_open_to_close_5d', 
                                       [df['gap_abs'] > gap_thresh, df['surprise_abs'] > surprise_cutoff], 
                                       'POS + big gap + big surprise → LONG T+5'))

ablation_df = pd.DataFrame([r for r in ablation if r is not None])
print("\n=== ABLATION: Filter Impact on T+5 Hold ===")
print(ablation_df.to_string(index=False))

## 4. Test Alternative Hypothesis: Negative Surprise Bounce (Contrarian)

Hypothesis: stocks that miss earnings and gap down often bounce over T+1 to T+5.

Filter: only large misses (negative surprise magnitude > threshold).

In [ ]:
# Test negative surprise bounce with filters
df_neg = df.copy()
df_neg['is_neg_surprise'] = df_neg['positive_eps_surprise'] == False

contrarian = []
contrarian.append(test_filtered_long(df_neg, 'is_neg_surprise', 'ret_open_to_close_1d', [], 'NEG → LONG T+1 (no filter)'))
contrarian.append(test_filtered_long(df_neg, 'is_neg_surprise', 'ret_open_to_close_5d', [], 'NEG → LONG T+5 (no filter)'))

# Filter: big gap down
contrarian.append(test_filtered_long(df_neg, 'is_neg_surprise', 'ret_open_to_close_5d', 
                                     [df_neg['gap_abs'] > gap_thresh], 
                                     'NEG + big gap → LONG T+5'))

if surprise_cutoff:
    contrarian.append(test_filtered_long(df_neg, 'is_neg_surprise', 'ret_open_to_close_5d', 
                                         [df_neg['surprise_abs'] > surprise_cutoff], 
                                         'NEG + big surprise → LONG T+5'))

contrarian_df = pd.DataFrame([r for r in contrarian if r is not None])
print("\n=== CONTRARIAN: Negative Surprise Bounce ===")
print(contrarian_df.to_string(index=False))

## 5. Visualize: Distribution of Returns by Strategy

In [ ]:
# Plot return distributions for POS vs NEG surprise, T+5 hold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pos_rets = pd.to_numeric(df[df['positive_eps_surprise'] == True]['ret_open_to_close_5d'], errors='coerce').dropna()
neg_rets = pd.to_numeric(df[df['positive_eps_surprise'] == False]['ret_open_to_close_5d'], errors='coerce').dropna()

axes[0].hist(pos_rets * 100, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[0].axvline(pos_rets.mean() * 100, color='darkgreen', linestyle='--', linewidth=2, label=f'Mean: {pos_rets.mean()*100:.2f}%')
axes[0].set_xlabel('Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('POS Surprise → LONG T+5 (Return Distribution)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(neg_rets * 100, bins=50, alpha=0.7, color='red', edgecolor='black')
axes[1].axvline(neg_rets.mean() * 100, color='darkred', linestyle='--', linewidth=2, label=f'Mean: {neg_rets.mean()*100:.2f}%')
axes[1].set_xlabel('Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('NEG Surprise → LONG T+5 (Return Distribution)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"POS surprise: mean={pos_rets.mean()*100:.2f}%, median={pos_rets.median()*100:.2f}%")
print(f"NEG surprise: mean={neg_rets.mean()*100:.2f}%, median={neg_rets.median()*100:.2f}%")

## 6. Next: Add Technical Context (EMA Regime)

To add EMA regime filtering, we need daily price data around each event.

Hypothesis: earnings signals work better when stock is already in uptrend (price > EMA50 > EMA200).

**TODO (next cell)**: 
- Fetch daily bars for each ticker around event dates
- Calculate EMA ribbons (20/50/200)
- Tag each event with regime (bull/bear/neutral)
- Re-run ablation with regime filter

In [ ]:
# Placeholder: EMA regime enrichment (requires fetching daily bars)
# This will be implemented in next iteration after we validate baseline results
print("⚠️  EMA regime filtering: pending implementation")
print("    Will fetch daily bars and calculate EMA(20/50/200) for each event.")

## 7. Summary: What Did We Find?

Review results and determine if there's a repeatable edge to build a companion around.

In [ ]:
print("="*80)
print("EARNINGS EDGE VALIDATION SUMMARY")
print("="*80)

print("\n1. BASELINE (no filters):")
print(baseline[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))

print("\n2. ABLATION (with filters):")
print(ablation_df[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))

print("\n3. CONTRARIAN (negative surprise bounce):")
print(contrarian_df[['strategy', 'n', 'net_mean', 'hit_rate']].to_string(index=False))

print("\n" + "="*80)
print("NEXT STEPS:")
print("- If net_mean > 1% and hit_rate > 52%, test on larger universe")
print("- Add EMA regime filter to improve hit rate")
print("- Add volume confirmation (spike on earnings day)")
print("- Build companion signal generator for upcoming earnings")
print("="*80)